# Transport Analytics Workbook

This workbook consolidates the archived project notebooks into one clean entry point.

## Included sections
1. Python/NumPy/Pandas/PySpark bootcamp
2. Data analytics pipeline
3. Literature and papers summary
4. Data science, big data, and SQL foundations
5. SQL active learning practice



---

## Source Notebook: `00_python_numpy_matplotlib_pandas_pyspark_bootcamp.ipynb`

**Section title:** 00 - Python, NumPy, Matplotlib, Pandas, and PySpark Bootcamp


# 00 - Python, NumPy, Matplotlib, Pandas, and PySpark Bootcamp

This is your **starting notebook** for the project.

Goal: go from fundamentals to production-ready understanding for this transport analytics project.

You will learn:
1. Python foundations
2. NumPy for vectorized numeric computing
3. Matplotlib for plots and visual debugging
4. Pandas for tabular/time-series analytics
5. PySpark foundations for big-data scale
6. Practice + active recall in each section

## How to use this notebook

- Run cells top to bottom.
- For practice cells, try first before reading the provided solution/check.
- Keep this notebook as your repeated training notebook.

## Project Explanation (Read This First)

This repository has **two data folders on purpose**.

### `data/` and `datasets/` (important)

`data/`:

- Main working storage for project data.
- Contains large/raw source files (like MTA) and processed outputs (`data/processed/...`).
- Think: pipeline working area.

`datasets/`:

- Curated external CSV datasets used as clean input sources.
- Usually smaller and more structured for analysis.
- Think: curated input package.

So:

- `data/` = raw + processed working data
- `datasets/` = curated dataset inputs

### Why both folders exist

- We keep **large/raw operational files** in `data/` because they change and can be very heavy.
- We keep **clean/selected research inputs** in `datasets/` so analysis notebooks can start quickly.
- We write pipeline outputs to `data/processed/` to avoid mixing raw and transformed data.

### Practical project flow

1. Raw/curated files are read from `data/` + `datasets/`.
2. Cleaning and schema harmonization happen in pipeline code.
3. Canonical fact tables are created in `data/processed/`.
4. SQL tables/views are applied in PostgreSQL (via pgAdmin web).
5. Analysis/forecasting notebooks use processed tables and SQL outputs.

### Where to look when you are new

- Start learning here: `00_python_numpy_matplotlib_pandas_pyspark_bootcamp.ipynb`
- Project analytics notebook: `01_data_analytics_pipeline.ipynb`
- SQL and DS foundations: `03_data_science_big_data_sql_foundations.ipynb`
- SQL practice: `04_sql_active_learning_practice.ipynb`
- PostgreSQL setup (pgAdmin web): `docs/POSTGRES_SETUP.md`

In [ ]:
# Standard imports used throughout the notebook.
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global notebook display settings.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

# Random generator with fixed seed for reproducibility.
rng = np.random.default_rng(42)

print("Environment ready.")

---
## Part A - Python Fundamentals

In [ ]:
# Variables and basic types.
city = "Paris"
riders_today = 15423
avg_delay_minutes = 3.7
is_weekend = False

# Collections.
line_codes = ["M1", "M4", "RER A"]
station_to_region = {"Chatelet": "Ile-de-France", "Times Sq": "NYC"}

print(type(city), type(riders_today), type(avg_delay_minutes), type(is_weekend))
print(line_codes)
print(station_to_region)

In [ ]:
# Control flow and looping.
weekly_counts = [12000, 11800, 12500, 13000, 12700, 9800, 9200]

weekday_total = 0
weekend_total = 0

for i, value in enumerate(weekly_counts):
    if i < 5:
        weekday_total += value
    else:
        weekend_total += value

print("weekday_total:", weekday_total)
print("weekend_total:", weekend_total)
print("weekend ratio:", round(weekend_total / (weekday_total + weekend_total), 3))

In [ ]:
# List comprehension and dictionary comprehension.
values = [12, 0, 4, 9, 0, 18, 7]

# Keep only non-zero values.
non_zero = [v for v in values if v != 0]

# Build index map for quick lookup.
idx_map = {idx: v for idx, v in enumerate(values)}

print("non_zero:", non_zero)
print("idx_map sample:", {k: idx_map[k] for k in [0, 3, 6]})

In [ ]:
# Functions with type hints and docstrings.
def pct_change(new_value: float, old_value: float) -> float:
    """Return percentage change from old_value to new_value."""
    if old_value == 0:
        return float("nan")
    return (new_value - old_value) / old_value

print("pct_change(13000, 12000):", round(pct_change(13000, 12000), 4))

In [ ]:
# Dataclass example for clear transport records.
@dataclass
class DemandPoint:
    date: str
    station: str
    riders: int

    def is_high_demand(self, threshold: int = 12000) -> bool:
        # Encapsulated logic improves readability and reuse.
        return self.riders >= threshold

p = DemandPoint(date="2024-01-10", station="Chatelet", riders=14200)
print(p)
print("high_demand:", p.is_high_demand())

### Python Practice (with checker)

Task:
- Write a function `moving_average(values, window)` returning a list of rolling means.
- Example: `[2, 4, 6, 8]` with `window=2` -> `[3.0, 5.0, 7.0]`

In [ ]:
# TODO: Replace this with your own implementation first.
def moving_average(values: list[float], window: int) -> list[float]:
    # Reference solution (you can overwrite).
    if window <= 0 or window > len(values):
        return []
    out = []
    for i in range(window - 1, len(values)):
        chunk = values[i - window + 1 : i + 1]
        out.append(sum(chunk) / window)
    return out

# Simple checker.
assert moving_average([2, 4, 6, 8], 2) == [3.0, 5.0, 7.0]
assert moving_average([1, 1, 1], 3) == [1.0]
print("Python practice check passed.")

### Active Recall - Python

Answer mentally first, then reveal.

In [ ]:
recall_cards_python = [
    ("What is a list comprehension?", "Compact syntax to create lists from iterables with optional filtering."),
    ("Why use a dataclass?", "To define lightweight classes with auto-generated init/repr and clearer structure."),
    ("Difference between list and tuple?", "List is mutable, tuple is immutable."),
    ("What does `if __name__ == '__main__'` do?", "Runs code only when file is executed directly, not imported."),
]

def draw_cards(cards, reveal=False):
    rows = []
    for q, a in cards:
        rows.append({"question": q, "answer": a if reveal else "(hidden)"})
    return pd.DataFrame(rows)

draw_cards(recall_cards_python, reveal=False)

---
## Part B - NumPy Foundations

In [ ]:
# Create arrays with explicit dtype and shape.
a = np.array([1, 2, 3, 4, 5], dtype=np.float64)
b = np.arange(12).reshape(3, 4)

print("a:", a)
print("a dtype:", a.dtype)
print("b:\n", b)
print("b shape:", b.shape)


In [ ]:
# Indexing, slicing, and boolean masks.
arr = np.array([10, 15, 8, 30, 22, 7, 14])

print("arr[0]:", arr[0])
print("arr[2:5]:", arr[2:5])

# Keep values above threshold.
mask = arr > 12
print("mask:", mask)
print("filtered:", arr[mask])

In [ ]:
# Vectorization and broadcasting.
base = np.array([100, 200, 300, 400])
growth_rate = 1.05

# Vectorized operation is faster than Python loops for large arrays.
next_period = base * growth_rate
print("next_period:", next_period)

# Broadcasting: add one offset per row.
matrix = np.arange(12).reshape(3, 4)
offsets = np.array([100, 200, 300]).reshape(3, 1)
print("matrix + offsets:\n", matrix + offsets)


In [ ]:
# Statistics and linear algebra basics.
x = rng.normal(loc=1000, scale=120, size=500)

print("mean:", round(x.mean(), 2))
print("std:", round(x.std(), 2))
print("p10/p50/p90:", np.percentile(x, [10, 50, 90]))

# Dot product example for weighted score.
weights = np.array([0.4, 0.35, 0.25])
features = np.array([0.8, 0.6, 0.9])
print("weighted score:", float(np.dot(weights, features)))

In [ ]:
# NumPy practice + checker.
# Task: normalize array with z-score: (x - mean) / std
def zscore(v: np.ndarray) -> np.ndarray:
    # Reference solution.
    return (v - v.mean()) / v.std()

test = np.array([1.0, 2.0, 3.0, 4.0])
out = zscore(test)
assert np.isclose(out.mean(), 0.0)
assert np.isclose(out.std(), 1.0)
print("NumPy practice check passed.")

---
## Part C - Matplotlib Foundations

In [ ]:
# Build synthetic daily demand series for plotting examples.
dates = pd.date_range("2024-01-01", periods=60, freq="D")
demand_fr = 12000 + 1200 * np.sin(np.linspace(0, 3 * np.pi, 60)) + rng.normal(0, 300, 60)
demand_us = 17000 + 1400 * np.sin(np.linspace(0, 3 * np.pi, 60) + 0.5) + rng.normal(0, 350, 60)

demo_df = pd.DataFrame({"date": dates, "fr": demand_fr, "us": demand_us})
demo_df.head()

In [ ]:
# Line chart for trend comparison.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(demo_df["date"], demo_df["fr"], label="Ile-de-France")
ax.plot(demo_df["date"], demo_df["us"], label="NYC")
ax.set_title("Daily demand trend")
ax.set_xlabel("Date")
ax.set_ylabel("Riders")
ax.legend()
plt.show()

In [ ]:
# Histogram + scatter + bar in one figure.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(demo_df["fr"], bins=15, color="#1f77b4")
axes[0].set_title("FR demand distribution")

axes[1].scatter(demo_df["fr"], demo_df["us"], alpha=0.7)
axes[1].set_title("FR vs US demand")
axes[1].set_xlabel("FR")
axes[1].set_ylabel("US")

weekly = demo_df.copy()
weekly["week"] = weekly["date"].dt.isocalendar().week.astype(int)
weekly = weekly.groupby("week", as_index=False)[["fr", "us"]].mean()
axes[2].bar(weekly["week"].astype(str), weekly["fr"], label="FR")
axes[2].set_title("Weekly avg FR demand")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

In [ ]:
# Matplotlib practice prompt:
# Build your own plot of rolling 7-day mean and compare with raw values.

temp = demo_df.copy()
temp["fr_roll7"] = temp["fr"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(temp["date"], temp["fr"], alpha=0.4, label="raw")
ax.plot(temp["date"], temp["fr_roll7"], linewidth=2, label="roll7")
ax.set_title("Practice solution: rolling mean")
ax.legend()
plt.show()

---
## Part D - Pandas Foundations

In [ ]:
# Series and DataFrame basics.
s = pd.Series([100, 120, 90], name="riders")

df = pd.DataFrame(
    {
        "station": ["A", "B", "C", "A", "B", "C"],
        "date": pd.date_range("2024-01-01", periods=6, freq="D"),
        "riders": [100, 130, 90, 115, 128, 97],
        "tickets": [80, 100, 70, 92, 101, 76],
    }
)

print(s)
df

In [ ]:
# Filtering, selecting, and assignment.
high = df[df["riders"] > 110].copy()
high["ratio_ticket_to_riders"] = high["tickets"] / high["riders"]

high

In [ ]:
# Groupby and aggregation patterns.
by_station = (
    df.groupby("station", as_index=False)
      .agg(
          mean_riders=("riders", "mean"),
          max_riders=("riders", "max"),
          sum_tickets=("tickets", "sum"),
      )
)

by_station

In [ ]:
# Merge / join patterns in pandas.
station_meta = pd.DataFrame(
    {
        "station": ["A", "B", "C"],
        "line": ["L1", "L1", "L2"],
        "region": ["FR", "FR", "US"],
    }
)

joined = df.merge(station_meta, on="station", how="left")
joined.head()

In [ ]:
# Time-series operations: resample and rolling.
ts = joined.set_index("date").sort_index()
weekly = ts.resample("W")["riders"].sum().to_frame("weekly_riders")
weekly["roll2"] = weekly["weekly_riders"].rolling(2, min_periods=1).mean()

weekly

In [ ]:
# Load real project CSV sample to connect training with project data.
root = Path("..").resolve()
sample_path = root / "datasets" / "Travel_titles_validations_in_Paris_and_suburbs.csv"

# Use nrows to keep notebook responsive.
real_sample = pd.read_csv(sample_path, nrows=20000)

# Basic cleanup for the sample.
real_sample["DATE"] = pd.to_datetime(real_sample["DATE"], dayfirst=True, errors="coerce")
real_sample["NB_VALID_NUM"] = (
    real_sample["NB_VALID"]
    .astype(str)
    .str.replace("Less than 5", "2", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

real_daily = real_sample.groupby("DATE", as_index=False)["NB_VALID_NUM"].sum()
real_daily.head()

In [ ]:
# Visualize real sample behavior.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(real_daily["DATE"], real_daily["NB_VALID_NUM"])
ax.set_title("Real project sample: daily validations")
ax.set_xlabel("Date")
ax.set_ylabel("Validations")
plt.show()

In [ ]:
# Pandas practice checker.
# Task: produce a DataFrame with station + total riders sorted descending.
def station_totals(input_df: pd.DataFrame) -> pd.DataFrame:
    # Reference solution.
    out = (
        input_df.groupby("station", as_index=False)["riders"]
        .sum()
        .rename(columns={"riders": "total_riders"})
        .sort_values("total_riders", ascending=False)
        .reset_index(drop=True)
    )
    return out

st = station_totals(df)
assert list(st.columns) == ["station", "total_riders"]
assert st["total_riders"].iloc[0] >= st["total_riders"].iloc[-1]
print("Pandas practice check passed.")
st

---
## Part E - PySpark Foundations (Big Data)

PySpark matters when data is too big for a single-machine pandas workflow.

In [ ]:
# Detect whether PySpark is available.
import importlib.util

spark_available = importlib.util.find_spec("pyspark") is not None
print("pyspark available:", spark_available)

In [ ]:
# PySpark section with safe fallback if pyspark is not installed.
if spark_available:
    # Import Spark only when available.
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    # Create local Spark session for demonstration.
    spark = SparkSession.builder.master("local[*]").appName("Bootcamp00").getOrCreate()

    # Convert pandas training DataFrame to Spark DataFrame.
    sdf = spark.createDataFrame(joined[["station", "date", "riders", "tickets", "line", "region"]])

    # Typical Spark transformations: select/filter/group/order.
    spark_result = (
        sdf.groupBy("region", "line")
        .agg(
            F.avg("riders").alias("avg_riders"),
            F.sum("riders").alias("sum_riders"),
        )
        .orderBy(F.desc("sum_riders"))
    )

    spark_result.show()
else:
    # Fallback explanation + pandas equivalent so notebook remains runnable.
    print("PySpark not installed in this environment.")
    print("Install later with: pip install pyspark")
    print("Equivalent pandas pattern shown below:")

    fallback = (
        joined.groupby(["region", "line"], as_index=False)
        .agg(avg_riders=("riders", "mean"), sum_riders=("riders", "sum"))
        .sort_values("sum_riders", ascending=False)
    )
    display(fallback)

In [ ]:
# Spark mental model: lazy transformations and actions.
spark_model = pd.DataFrame(
    {
        "concept": ["Transformation", "Action", "Lazy execution", "Partitioning"],
        "example": ["select, filter, withColumn", "count, show, collect", "Plan executes on action", "Parallel chunks of data"],
        "why_it_matters": [
            "Build pipeline steps",
            "Trigger computation",
            "Optimize distributed execution",
            "Scale workload over cluster",
        ],
    }
)

spark_model

### PySpark Practice Prompt

Write Spark code (or pandas fallback) to compute:
- total riders per region per day
- 7-day rolling average per region

You can use the patterns from previous sections.

---
## Part F - Mini Project Workflow (from this repository)

In [ ]:
# Build a mini end-to-end pipeline using real sampled project data.
root = Path("..").resolve()
idfm_path = root / "datasets" / "idfm_validations_surface.csv"

# The file uses ';' separator.
idfm = pd.read_csv(idfm_path, sep=";", nrows=50000)

# Basic cleaning for project-ready columns.
idfm["JOUR"] = pd.to_datetime(idfm["JOUR"], errors="coerce")
idfm["NB_VALD"] = pd.to_numeric(idfm["NB_VALD"], errors="coerce")

# Daily region-like aggregation at line level.
idfm_daily = (
    idfm.groupby(["JOUR", "LIBELLE_LIGNE"], as_index=False)["NB_VALD"]
    .sum()
    .rename(columns={"JOUR": "date", "LIBELLE_LIGNE": "location_name", "NB_VALD": "value"})
)

# Add training-style features.
idfm_daily = idfm_daily.sort_values(["location_name", "date"])
idfm_daily["dow"] = idfm_daily["date"].dt.dayofweek
idfm_daily["lag_1"] = idfm_daily.groupby("location_name")["value"].shift(1)
idfm_daily["roll7"] = idfm_daily.groupby("location_name")["value"].transform(lambda s: s.rolling(7, min_periods=2).mean())

idfm_daily.head()

In [ ]:
# Plot one location to understand trend + rolling behavior.
top_line = (
    idfm_daily.groupby("location_name", as_index=False)["value"].sum()
    .sort_values("value", ascending=False)
    .iloc[0]["location_name"]
)

plot_df = idfm_daily[idfm_daily["location_name"] == top_line].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(plot_df["date"], plot_df["value"], alpha=0.45, label="raw")
ax.plot(plot_df["date"], plot_df["roll7"], linewidth=2, label="roll7")
ax.set_title(f"Mini project view - {top_line}")
ax.legend()
plt.show()

---
## Part G - Active Learning and Revision System

In [ ]:
# Unified recall deck across all topics.
recall_deck = pd.DataFrame(
    [
        ("Python", "What problem do type hints solve?", "They improve readability, tooling, and early error detection."),
        ("NumPy", "What is broadcasting?", "Applying operations on arrays with compatible shapes without manual loops."),
        ("Matplotlib", "When to use scatter vs line?", "Scatter for relationship points; line for ordered trend/time series."),
        ("Pandas", "What does groupby-agg do?", "Splits data by keys then computes summary metrics."),
        ("PySpark", "What is lazy execution?", "Spark defers computation until an action is called."),
        ("Project", "Why create lag features?", "To give models access to past behavior for forecasting."),
    ],
    columns=["topic", "question", "answer"],
)

def practice_cards(deck: pd.DataFrame, n: int = 4, reveal: bool = False, seed: int = 0) -> pd.DataFrame:
    sample = deck.sample(n=min(n, len(deck)), random_state=seed).reset_index(drop=True)
    if not reveal:
        sample = sample.copy()
        sample["answer"] = "(hidden)"
    return sample

practice_cards(recall_deck, n=5, reveal=False, seed=2)

In [ ]:
# Mini multiple-choice quiz with auto-scoring.
quiz = [
    {
        "q": "Which library is best for single-machine tabular analytics?",
        "options": ["A) NumPy", "B) Pandas", "C) Matplotlib", "D) PySpark SQL only"],
        "answer": "B",
    },
    {
        "q": "Which operation triggers Spark execution?",
        "options": ["A) filter", "B) withColumn", "C) show", "D) select"],
        "answer": "C",
    },
    {
        "q": "What does a 7-day rolling mean help with?",
        "options": ["A) Random shuffling", "B) Smoothing noise", "C) Encoding strings", "D) SQL parsing"],
        "answer": "B",
    },
]

user_answers = ["B", "C", "B"]  # Replace with your answers.

score = 0
for i, item in enumerate(quiz):
    print(f"Q{i+1}: {item['q']}")
    print(" ".join(item["options"]))
    print("Your answer:", user_answers[i], "| Correct:", item["answer"])
    if user_answers[i].upper() == item["answer"]:
        score += 1
    print("---")

print(f"Score: {score}/{len(quiz)}")

## Suggested path after this notebook

1. `01_data_analytics_pipeline.ipynb`
2. `02_papers_summary.ipynb`
3. `03_data_science_big_data_sql_foundations.ipynb`
4. `04_sql_active_learning_practice.ipynb`

You can revisit this `00` notebook anytime for revision.


---

## Source Notebook: `01_data_analytics_pipeline.ipynb`

**Section title:** Big Data Project - Complete Data Analytics Notebook


# Big Data Project - Complete Data Analytics Notebook

This notebook analyzes all available project datasets under `datasets/` and `data/`, checks how columns can be combined,
and builds a reusable pipeline-ready fact table.

It is designed to be runnable on this repository as-is, with a **sample mode** enabled by default for very large files.

## What this notebook does

1. Loads and profiles datasets from both `datasets/` and `data/`.
2. Normalizes date, identifier, and count columns across formats (CSV/TXT, comma/semicolon/tab, mixed encodings).
3. Evaluates which datasets can be combined directly (station-level vs date-level).
4. Builds a canonical fact table for downstream modeling.
5. Adds feature-engineering blocks for demand forecasting and anomaly detection.
6. Demonstrates a future-proof enrichment step for weather and holiday datasets.

## Project Data Explanation

Before analysis, understand folder roles:

`data/`:

- Main working storage for project data.
- Contains large/raw source files (like MTA) and processed outputs (`data/processed/...`).
- Think: pipeline working area.

`datasets/`:

- Curated external CSV datasets used as clean input sources.
- Usually smaller and more structured for analysis.
- Think: curated input package.

So:

- `data/` = raw + processed working data
- `datasets/` = curated dataset inputs

In this notebook, we read both, then normalize them into a canonical fact table.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Paths and execution controls
ROOT = Path("..").resolve()
DATASETS_DIR = ROOT / "datasets"
DATA_DIR = ROOT / "data"
IDF_DIR = DATA_DIR / "Ile_de_france"
MTA_DIR = DATA_DIR / "soroosh_MTA"

# Sample mode keeps notebook fast and memory-safe for local execution.
USE_SAMPLE_MODE = True
MTA_SAMPLE_ROWS = 300_000
MAX_IDF_FILES = 6
MAX_ROWS_PER_IDF_FILE = 250_000

print("ROOT:", ROOT)
print("USE_SAMPLE_MODE:", USE_SAMPLE_MODE)

In [ ]:
def detect_separator(file_path: Path) -> str:
    # Infer delimiter from first line
    header = file_path.open("rb").readline().decode("latin1", errors="ignore")
    counts = {",": header.count(","), ";": header.count(";"), "	": header.count("	")}
    return max(counts, key=counts.get)


def read_table_smart(file_path: Path, nrows: int | None = None) -> pd.DataFrame:
    # Read CSV/TXT with fallback encodings
    sep = detect_separator(file_path)
    encodings = ["utf-8-sig", "utf-8", "latin1", "cp1252", "utf-16"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(file_path, sep=sep, encoding=enc, nrows=nrows, low_memory=False)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Failed to read {file_path.name}: {last_error}")


def clean_numeric(series: pd.Series) -> pd.Series:
    # Normalize mixed numeric formats: spaces, comma decimals, text labels
    s = series.astype(str).str.strip()
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace("Less than 5", "2", regex=False)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "?": np.nan})
    return pd.to_numeric(s, errors="coerce")


def parse_any_date(series: pd.Series) -> pd.Series:
    # Parse mixed date formats
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

In [ ]:
# Inventory project data files
file_list = sorted(
    list(DATASETS_DIR.glob("*.csv"))
    + list(IDF_DIR.glob("*.csv"))
    + list(IDF_DIR.glob("data-rf-*/*NB_FER*.txt"))
    + list(IDF_DIR.glob("data-rf-*/*NB_FER*.csv"))
    + list(IDF_DIR.glob("data-rf-*/*PROFIL*.txt"))
    + list(IDF_DIR.glob("data-rf-*/*PROFIL*.csv"))
    + list((IDF_DIR / "data-rf-2020" / "data-rf-2020").glob("*.txt"))
    + list(MTA_DIR.glob("*.csv"))
)

inventory = pd.DataFrame(
    {
        "file": [f.name for f in file_list],
        "relative_path": [str(f.relative_to(ROOT)) for f in file_list],
        "size_mb": [round(f.stat().st_size / (1024 ** 2), 2) for f in file_list],
    }
).sort_values("size_mb", ascending=False)

inventory.head(20)

In [ ]:
# Load core datasets (moderate size)
regularities_fr = pd.read_csv(DATASETS_DIR / "Regularities_by_liaisons_Trains_France.csv")
travel_titles = pd.read_csv(DATASETS_DIR / "Travel_titles_validations_in_Paris_and_suburbs.csv")
idfm_surface = read_table_smart(DATASETS_DIR / "idfm_validations_surface.csv")
tgv_monthly = read_table_smart(IDF_DIR / "regularite-mensuelle-tgv-aqst.csv")

core_datasets = {
    "regularities_fr": regularities_fr,
    "travel_titles": travel_titles,
    "idfm_surface": idfm_surface,
    "tgv_monthly": tgv_monthly,
}

profile_rows = []
for name, df in core_datasets.items():
    profile_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_pct": round(df.isna().mean().mean() * 100, 2),
        }
    )

pd.DataFrame(profile_rows).sort_values("rows", ascending=False)

In [ ]:
# Normalize key fields for analytics
travel_titles = travel_titles.copy()
travel_titles["date"] = parse_any_date(travel_titles["DATE"])
travel_titles["validations"] = clean_numeric(travel_titles["NB_VALID"])

idfm_surface = idfm_surface.copy()
idfm_surface["date"] = parse_any_date(idfm_surface["JOUR"])
idfm_surface["validations"] = clean_numeric(idfm_surface["NB_VALD"])

regularities_fr = regularities_fr.copy()
regularities_fr["period"] = pd.to_datetime(regularities_fr["Period"], format="%Y-%m", errors="coerce")
regularities_fr["late_trains_arrival"] = clean_numeric(regularities_fr["Number of trains late on arrival"])

tgv_monthly = tgv_monthly.copy()
tgv_monthly["period"] = pd.to_datetime(tgv_monthly["Date"], format="%Y-%m", errors="coerce")
tgv_monthly["planned_trains"] = clean_numeric(tgv_monthly["Nombre de circulations prévues"])

daily_travel = (
    travel_titles.dropna(subset=["date"])
    .groupby("date", as_index=False)["validations"]
    .sum()
    .assign(source="travel_titles_paris")
)

daily_surface = (
    idfm_surface.dropna(subset=["date"])
    .groupby("date", as_index=False)["validations"]
    .sum()
    .assign(source="idfm_surface")
)

monthly_tgv = (
    tgv_monthly.dropna(subset=["period"])
    .groupby("period", as_index=False)["planned_trains"]
    .sum()
    .rename(columns={"period": "date", "planned_trains": "validations"})
    .assign(source="tgv_planned_trains")
)

core_daily = pd.concat([daily_travel, daily_surface, monthly_tgv], ignore_index=True)
core_daily.head()

In [ ]:
# Trend visualization across core datasets
fig, ax = plt.subplots(figsize=(13, 5))
for src, g in core_daily.groupby("source"):
    g = g.sort_values("date")
    ax.plot(g["date"], g["validations"], label=src, linewidth=1.8)
ax.set_title("Daily/Monthly Demand Signals Across Core Datasets")
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.legend()
plt.show()

# Top stations in travel_titles dataset
top_stations = (
    travel_titles.groupby("STATION_NAME", as_index=False)["validations"]
    .sum()
    .sort_values("validations", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_stations["STATION_NAME"], top_stations["validations"])
ax.set_title("Top 15 Stations by Total Validations (Travel Titles)")
ax.set_ylabel("Validations")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()

In [ ]:
# Load Ile-de-France NB_FER files (daily validations by stop)
def load_idf_nb_files(base_dir: Path, sample_mode: bool = True) -> pd.DataFrame:
    nb_files = sorted(base_dir.glob("data-rf-*/*NB_FER*.txt")) + sorted(base_dir.glob("data-rf-*/*NB_FER*.csv"))
    nb_files += sorted((base_dir / "data-rf-2020" / "data-rf-2020").glob("*NB_FER*.txt"))

    if sample_mode:
        nb_files = nb_files[-MAX_IDF_FILES:]

    frames = []
    for fp in nb_files:
        nrows = MAX_ROWS_PER_IDF_FILE if sample_mode else None
        df = read_table_smart(fp, nrows=nrows)
        df.columns = [c.strip() for c in df.columns]

        date_col = "JOUR" if "JOUR" in df.columns else None
        id_col = "ID_ZDC" if "ID_ZDC" in df.columns else ("ID_REFA_LDA" if "ID_REFA_LDA" in df.columns else ("lda" if "lda" in df.columns else None))
        val_col = "NB_VALD" if "NB_VALD" in df.columns else None

        if date_col is None or val_col is None:
            continue

        out = pd.DataFrame(
            {
                "date": parse_any_date(df[date_col]),
                "station_id": df[id_col].astype(str) if id_col else np.nan,
                "station_name": df.get("LIBELLE_ARRET", np.nan),
                "ticket_category": df.get("CATEGORIE_TITRE", np.nan),
                "validations": clean_numeric(df[val_col]),
                "source_file": fp.name,
            }
        )
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=["date", "station_id", "station_name", "ticket_category", "validations", "source_file"])
    return pd.concat(frames, ignore_index=True)


idf_nb = load_idf_nb_files(IDF_DIR, sample_mode=USE_SAMPLE_MODE)
idf_nb_profile = {
    "rows": len(idf_nb),
    "stations": idf_nb["station_id"].nunique(dropna=True),
    "min_date": str(idf_nb["date"].min()),
    "max_date": str(idf_nb["date"].max()),
}
idf_nb_profile

In [ ]:
# Load Ile-de-France PROFIL files (hourly profile percentages)
def load_idf_profil_files(base_dir: Path, sample_mode: bool = True) -> pd.DataFrame:
    profil_files = sorted(base_dir.glob("data-rf-*/*PROFIL*.txt")) + sorted(base_dir.glob("data-rf-*/*PROFIL*.csv"))
    profil_files += sorted((base_dir / "data-rf-2020" / "data-rf-2020").glob("*PROFIL*.txt"))

    if sample_mode:
        profil_files = profil_files[-MAX_IDF_FILES:]

    frames = []
    for fp in profil_files:
        nrows = MAX_ROWS_PER_IDF_FILE if sample_mode else None
        df = read_table_smart(fp, nrows=nrows)
        df.columns = [c.strip() for c in df.columns]

        pct_col = "pourc_validations" if "pourc_validations" in df.columns else ("Pourcentage_validations" if "Pourcentage_validations" in df.columns else None)
        id_col = "ID_ZDC" if "ID_ZDC" in df.columns else ("ID_REFA_LDA" if "ID_REFA_LDA" in df.columns else ("lda" if "lda" in df.columns else None))

        if pct_col is None:
            continue

        out = pd.DataFrame(
            {
                "station_id": df[id_col].astype(str) if id_col else np.nan,
                "cat_jour": df.get("CAT_JOUR", np.nan),
                "hour_bin": df.get("TRNC_HORR_60", np.nan),
                "pct_validations": clean_numeric(df[pct_col]),
                "source_file": fp.name,
            }
        )
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=["station_id", "cat_jour", "hour_bin", "pct_validations", "source_file"])
    return pd.concat(frames, ignore_index=True)


idf_profil = load_idf_profil_files(IDF_DIR, sample_mode=USE_SAMPLE_MODE)
idf_profil.head()

In [ ]:
# Check which columns allow direct combination across datasets
compatibility = pd.DataFrame(
    [
        {
            "dataset": "travel_titles_paris",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idfm_surface",
            "date": True,
            "station_id": False,
            "line_id": True,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idf_nb_fer",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idf_profil_fer",
            "date": False,
            "station_id": True,
            "line_id": False,
            "hour_bin": True,
            "ticket_category": False,
            "count_metric": False,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "mta_hourly",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": True,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "NYC",
        },
        {
            "dataset": "tgv_regularities",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": False,
            "count_metric": True,
            "geo_area": "France",
        },
    ]
)

display(compatibility)

matrix_cols = ["date", "station_id", "line_id", "hour_bin", "ticket_category", "count_metric"]
matrix = compatibility.set_index("dataset")[matrix_cols].astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(matrix.values, aspect="auto")
ax.set_xticks(range(len(matrix_cols)))
ax.set_xticklabels(matrix_cols, rotation=45, ha="right")
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_title("Join-Key Compatibility Matrix (1=available)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Canonical fact table design

To support current analysis and future integration, we standardize all datasets to:

- `date`
- `region`
- `location_id`
- `location_name`
- `metric_type` (`validations`, `ridership`, `delay`, ...)
- `value`
- `source`

In [ ]:
def aggregate_mta_daily(mta_csv: Path, sample_mode: bool = True) -> pd.DataFrame:
    usecols = [
        "transit_timestamp",
        "station_complex_id",
        "station_complex",
        "borough",
        "payment_method",
        "fare_class_category",
        "ridership",
        "transfers",
    ]

    read_kwargs = dict(usecols=usecols, low_memory=False)
    if sample_mode:
        read_kwargs["nrows"] = MTA_SAMPLE_ROWS

    mta = pd.read_csv(mta_csv, **read_kwargs)
    mta["date"] = pd.to_datetime(mta["transit_timestamp"], errors="coerce")
    mta["date"] = mta["date"].dt.floor("D")
    mta["ridership"] = pd.to_numeric(mta["ridership"], errors="coerce")
    mta["transfers"] = pd.to_numeric(mta["transfers"], errors="coerce")

    daily = (
        mta.dropna(subset=["date"])
        .groupby(["date", "borough", "station_complex_id", "station_complex"], as_index=False)[["ridership", "transfers"]]
        .sum()
    )
    return daily


mta_path = MTA_DIR / "MTA_Subway_Hourly_Ridership__2020-2024.csv"
mta_daily = aggregate_mta_daily(mta_path, sample_mode=USE_SAMPLE_MODE)
mta_daily.head()

In [ ]:
# Build canonical facts from each source
facts = []

tt_fact = pd.DataFrame(
    {
        "date": travel_titles["date"],
        "region": "Ile-de-France",
        "location_id": travel_titles["ID_REFA_LDA"].astype(str),
        "location_name": travel_titles["STATION_NAME"],
        "metric_type": "validations",
        "value": travel_titles["validations"],
        "source": "travel_titles_paris",
    }
)
facts.append(tt_fact)

sf_fact = pd.DataFrame(
    {
        "date": idfm_surface["date"],
        "region": "Ile-de-France",
        "location_id": idfm_surface.get("ID_GROUPOFLINES", pd.Series([np.nan] * len(idfm_surface))).astype(str),
        "location_name": idfm_surface.get("LIBELLE_LIGNE", pd.Series([np.nan] * len(idfm_surface))),
        "metric_type": "validations",
        "value": idfm_surface["validations"],
        "source": "idfm_surface",
    }
)
facts.append(sf_fact)

if not idf_nb.empty:
    nb_fact = pd.DataFrame(
        {
            "date": idf_nb["date"],
            "region": "Ile-de-France",
            "location_id": idf_nb["station_id"].astype(str),
            "location_name": idf_nb["station_name"],
            "metric_type": "validations",
            "value": idf_nb["validations"],
            "source": "idf_nb_fer",
        }
    )
    facts.append(nb_fact)

mta_fact = pd.DataFrame(
    {
        "date": mta_daily["date"],
        "region": mta_daily["borough"].fillna("NYC"),
        "location_id": mta_daily["station_complex_id"].astype(str),
        "location_name": mta_daily["station_complex"],
        "metric_type": "ridership",
        "value": mta_daily["ridership"],
        "source": "mta_hourly_agg_daily",
    }
)
facts.append(mta_fact)

fact_table = pd.concat(facts, ignore_index=True)
fact_table["date"] = pd.to_datetime(fact_table["date"], errors="coerce").dt.floor("D")
fact_table["value"] = pd.to_numeric(fact_table["value"], errors="coerce")
fact_table = fact_table.dropna(subset=["date", "value"])

print("Fact table rows:", len(fact_table))
fact_table.head()

In [ ]:
daily_fact = (
    fact_table.groupby(["date", "region", "source", "metric_type"], as_index=False)["value"]
    .sum()
    .sort_values(["source", "region", "date"])
)

daily_fact.head(10)

In [ ]:
def add_time_features(df: pd.DataFrame, group_cols=("region", "source", "metric_type")) -> pd.DataFrame:
    out = df.copy().sort_values([*group_cols, "date"])
    out["day_of_week"] = out["date"].dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["month"] = out["date"].dt.month
    out["week_of_year"] = out["date"].dt.isocalendar().week.astype(int)

    grouped = out.groupby(list(group_cols))["value"]
    out["lag_1"] = grouped.shift(1)
    out["lag_7"] = grouped.shift(7)
    out["rolling_7_mean"] = grouped.transform(lambda s: s.rolling(7, min_periods=3).mean())
    out["rolling_7_std"] = grouped.transform(lambda s: s.rolling(7, min_periods=3).std())
    out["pct_change_1"] = grouped.pct_change()
    out["zscore_30"] = grouped.transform(
        lambda s: (s - s.rolling(30, min_periods=10).mean()) / s.rolling(30, min_periods=10).std()
    )
    return out


featured = add_time_features(daily_fact)
featured.tail()

In [ ]:
selected = featured[featured["source"].isin(["travel_titles_paris", "idfm_surface", "mta_hourly_agg_daily"])]

weekday_profile = (
    selected.groupby(["source", "day_of_week"], as_index=False)["value"].mean()
    .rename(columns={"value": "avg_value"})
)

fig, ax = plt.subplots(figsize=(10, 5))
for src, g in weekday_profile.groupby("source"):
    ax.plot(g["day_of_week"], g["avg_value"], marker="o", label=src)
ax.set_title("Average Demand by Day of Week")
ax.set_xlabel("Day of week (0=Mon)")
ax.set_ylabel("Average count")
ax.legend()
plt.show()

anomaly_view = selected.dropna(subset=["zscore_30"]).copy()
anomaly_view["is_anomaly"] = (anomaly_view["zscore_30"].abs() >= 2.5)
anomaly_rate = (
    anomaly_view.groupby("source", as_index=False)["is_anomaly"].mean()
    .rename(columns={"is_anomaly": "anomaly_rate"})
)
anomaly_rate

## Future integration pipeline: Weather + Holidays

The following block shows how to enrich the canonical fact table with external datasets.
For now we use synthetic examples with the same schema expected from real providers.

In [ ]:
date_span = pd.date_range(daily_fact["date"].min(), daily_fact["date"].max(), freq="D")

holiday_demo = pd.DataFrame(
    {
        "date": date_span,
        "country": np.where(date_span.month <= 6, "FR", "US"),
        "is_holiday": ((date_span.day == 1) & (date_span.month.isin([1, 5, 7, 11]))).astype(int),
        "holiday_name": np.where((date_span.day == 1) & (date_span.month == 1), "New Year", ""),
    }
)

rng = np.random.default_rng(42)
weather_demo = pd.DataFrame(
    {
        "date": np.repeat(date_span, 2),
        "country": ["FR", "US"] * len(date_span),
        "mean_temp_c": rng.normal(loc=14, scale=9, size=len(date_span) * 2),
        "precip_mm": np.clip(rng.gamma(shape=1.8, scale=2.0, size=len(date_span) * 2), 0, 35),
    }
)


def attach_external_context(df: pd.DataFrame, holidays: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["country"] = np.where(out["region"].str.contains("Ile|France", case=False, na=False), "FR", "US")
    out = out.merge(holidays, on=["date", "country"], how="left")
    out = out.merge(weather, on=["date", "country"], how="left")
    out["is_holiday"] = out["is_holiday"].fillna(0).astype(int)
    return out


enriched = attach_external_context(featured, holiday_demo, weather_demo)
enriched[["date", "region", "source", "value", "is_holiday", "mean_temp_c", "precip_mm"]].head()

In [ ]:
# Pipeline blueprint visualization
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")

boxes = [
    (0.05, 0.55, "Raw data\n(datasets/ + data/)"),
    (0.26, 0.55, "Normalization\n(schema + types)"),
    (0.47, 0.55, "Canonical fact table\n(date, region, metric, value)"),
    (0.68, 0.55, "Enrichment\n(weather + holidays)"),
    (0.86, 0.55, "Models\nforecast + anomalies"),
]

for x, y, label in boxes:
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", fc="#e8f0fe", ec="#2c3e50"),
    )

for i in range(len(boxes) - 1):
    x1, y1, _ = boxes[i]
    x2, y2, _ = boxes[i + 1]
    ax.annotate("", xy=(x2 - 0.06, y2), xytext=(x1 + 0.08, y1), arrowprops=dict(arrowstyle="->", lw=1.8))

ax.set_title("Big Data Project Pipeline (Current + Future Extensions)")
plt.show()

In [ ]:
# Persist outputs for future modeling scripts
processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

daily_fact.to_csv(processed_dir / "daily_fact_table.csv", index=False)
enriched.to_csv(processed_dir / "daily_fact_table_enriched_demo.csv", index=False)

print("Saved:")
print("-", processed_dir / "daily_fact_table.csv")
print("-", processed_dir / "daily_fact_table_enriched_demo.csv")

## Key conclusions

- **Best direct joins inside Ile-de-France:** `travel_titles` + `idf_nb_fer` (date + station id).
- **`idfm_surface`** is best merged at **date/line level**, not station level.
- **MTA** is highly valuable for high-frequency modeling, but it should stay in a separate geographic branch (`US`) and join with shared external context (weather/holiday/calendar).
- The canonical fact table created here is ready for forecasting and anomaly pipelines.

To run full scale processing, set:

```python
USE_SAMPLE_MODE = False
```


---

## Source Notebook: `02_papers_summary.ipynb`

**Section title:** Big Data Project - Papers and Docs Summary Notebook


# Big Data Project - Papers and Docs Summary Notebook

This notebook summarizes local papers under `docs/papers/` and links them to concrete modeling decisions for this project.
It includes reproducible code for metadata extraction, paper cataloging, and planning research-to-implementation steps.

## Scope and method

- Source files: `docs/papers/**/*.pdf` and `docs/papers/citations.txt`
- Since internet is not required for this notebook, summaries are built from:
  - local citation metadata
  - embedded PDF metadata/keywords/section titles when available
  - paper titles and domain context
- Every result below is reproducible from local files.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path("..").resolve()
PAPERS_DIR = ROOT / "docs" / "papers"
CITATIONS_FILE = PAPERS_DIR / "citations.txt"

In [ ]:
# Parse citations file
text = CITATIONS_FILE.read_text(encoding="utf-8", errors="ignore")

lines = text.splitlines()
current_team = None
rows = []

i = 0
while i < len(lines):
    line = lines[i].strip()
    team_match = re.match(r"^\[(.+?)\]$", line)
    if team_match:
        current_team = team_match.group(1)
        i += 1
        continue

    entry_match = re.match(r"^(\d+)\.\s*(.+)$", line)
    if entry_match:
        idx = int(entry_match.group(1))
        citation = entry_match.group(2).strip()
        url = ""
        if i + 1 < len(lines) and "URL:" in lines[i + 1]:
            url = lines[i + 1].split("URL:", 1)[1].strip()
            i += 1

        year_match = re.search(r"\((19|20)\d{2}\)", citation)
        year = int(year_match.group(0).strip("()")) if year_match else np.nan

        rows.append({"id": idx, "team": current_team, "citation": citation, "year": year, "url": url})
    i += 1

citations_df = pd.DataFrame(rows).sort_values("id")
citations_df

In [ ]:
# Extract lightweight metadata directly from PDF bytes (no external packages required)
pdf_files = sorted(PAPERS_DIR.glob("*/*.pdf"))


def decode_pdf_unicode_escapes(raw: str) -> str:
    # Convert patterns like \000A\000b... from PDF bookmarks to readable text.
    cleaned = raw.replace("\\376\\377", "")
    pieces = re.findall(r"\\([0-9A-Fa-f]{4})", cleaned)
    if pieces:
        try:
            return "".join(chr(int(p, 16)) for p in pieces)
        except Exception:
            return raw
    return raw


def extract_pdf_metadata(path: Path) -> dict:
    data = path.read_bytes().decode("latin1", errors="ignore")

    title = None
    title_candidates = re.findall(r"/Title\(([^\)]{5,300})\)", data)
    if title_candidates:
        for cand in reversed(title_candidates):
            low = cand.lower()
            if "fig" not in low and "table" not in low and "introduction" not in low:
                title = decode_pdf_unicode_escapes(cand)
                break
        if title is None:
            title = decode_pdf_unicode_escapes(title_candidates[-1])

    author_match = re.search(r"/Author\(([^\)]{2,200})\)", data)
    author = author_match.group(1) if author_match else None

    doi_match = re.search(r"10\.\d{4,9}/[-._;()/:A-Za-z0-9]+", data)
    doi = doi_match.group(0) if doi_match else None

    kw_match = re.search(r"<pdf:Keywords>(.*?)</pdf:Keywords>", data, flags=re.DOTALL)
    keywords = kw_match.group(1).strip() if kw_match else None

    section_titles = re.findall(r"<<\s*/Title\(([^\)]{4,250})\)", data)
    section_titles = [decode_pdf_unicode_escapes(s).strip() for s in section_titles]
    section_titles = [s for s in section_titles if s and len(s) < 120]

    pages_est = len(re.findall(r"/Type/Page\b", data))

    return {
        "paper_file": path.name,
        "paper_path": str(path.relative_to(ROOT)),
        "folder": path.parent.name,
        "title_meta": title,
        "author_meta": author,
        "doi_meta": doi,
        "keywords_meta": keywords,
        "pages_est": pages_est,
        "section_samples": section_titles[:12],
    }


pdf_meta_df = pd.DataFrame([extract_pdf_metadata(p) for p in pdf_files])
pdf_meta_df[["folder", "paper_file", "title_meta", "doi_meta", "pages_est"]]

In [ ]:
# Literature summary table aligned to this project
summary_rows = [
    {
        "paper": "ASTIR: Spatio-Temporal Data Mining for Crowd Flow Prediction",
        "year": 2019,
        "theme": "forecasting",
        "method_family": "deep spatio-temporal",
        "main_signal": "grid/time crowd flow",
        "project_use": "Inspire multi-scale temporal blocks for ridership forecasting",
        "pipeline_stage": "modeling",
        "priority": 5,
    },
    {
        "paper": "Model-Adaptive Event Triggering for Monitoring Recurrent Mobility Patterns in Public Transport",
        "year": 2023,
        "theme": "anomaly monitoring",
        "method_family": "event-triggered monitoring",
        "main_signal": "recurrent mobility patterns",
        "project_use": "Design online alerting for abnormal daily demand",
        "pipeline_stage": "monitoring",
        "priority": 5,
    },
    {
        "paper": "Time-series clustering - A decade review",
        "year": 2015,
        "theme": "representation learning",
        "method_family": "clustering survey",
        "main_signal": "time-series similarity",
        "project_use": "Guide clustering choices for station archetypes",
        "pipeline_stage": "analysis",
        "priority": 4,
    },
    {
        "paper": "Benefits from a new transit line",
        "year": 2025,
        "theme": "policy impact",
        "method_family": "before-after ridership analysis",
        "main_signal": "light rail usage intensity",
        "project_use": "Evaluate interventions using usage intensity segments",
        "pipeline_stage": "evaluation",
        "priority": 4,
    },
    {
        "paper": "Unveiling mobility patterns beyond home/work activities",
        "year": 2024,
        "theme": "mobility behavior",
        "method_family": "topic modeling",
        "main_signal": "smart card + land-use",
        "project_use": "Add activity-pattern latent features to forecasting",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "A Two-Stage Trip Inference Model",
        "year": 2025,
        "theme": "trip purpose inference",
        "method_family": "two-stage inference",
        "main_signal": "regular user trajectories",
        "project_use": "Infer purpose labels for demand segmentation",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "Optimizing Urban Mobility Through Complex Network Analysis and Big Data from Smart Cards",
        "year": 2025,
        "theme": "network analytics",
        "method_family": "complex networks",
        "main_signal": "OD/network topology",
        "project_use": "Add station centrality and robustness indicators",
        "pipeline_stage": "analysis",
        "priority": 3,
    },
    {
        "paper": "Mining Smart Card Data for Transit Riders' Travel Patterns",
        "year": 2013,
        "theme": "travel pattern mining",
        "method_family": "data mining classification",
        "main_signal": "trip chains and rider profiles",
        "project_use": "Create rider-type features from repeated behavior",
        "pipeline_stage": "feature engineering",
        "priority": 4,
    },
    {
        "paper": "Identifying Human Mobility Patterns Using Smart Card Data",
        "year": 2022,
        "theme": "mobility pattern detection",
        "method_family": "pattern discovery",
        "main_signal": "longitudinal smart card behaviors",
        "project_use": "Improve segmentation of recurrent temporal profiles",
        "pipeline_stage": "analysis",
        "priority": 4,
    },
    {
        "paper": "Combining Smart Card Data and Household Travel Survey",
        "year": 2015,
        "theme": "data fusion",
        "method_family": "survey + smart card integration",
        "main_signal": "jobs-housing and socio-spatial links",
        "project_use": "Fuse external socio-economic context in model evaluation",
        "pipeline_stage": "enrichment",
        "priority": 3,
    },
]

summary_df = pd.DataFrame(summary_rows).sort_values(["priority", "year"], ascending=[False, False])
summary_df

In [ ]:
# Research landscape charts
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

by_theme = summary_df["theme"].value_counts().sort_values(ascending=True)
axes[0].barh(by_theme.index, by_theme.values)
axes[0].set_title("Papers by Theme")

by_year = summary_df.groupby("year", as_index=False).size()
axes[1].plot(by_year["year"], by_year["size"], marker="o")
axes[1].set_title("Publications Over Time")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")

by_stage = summary_df["pipeline_stage"].value_counts().sort_values(ascending=True)
axes[2].barh(by_stage.index, by_stage.values, color="#2ca02c")
axes[2].set_title("Coverage by Pipeline Stage")

plt.tight_layout()
plt.show()

In [ ]:
# Keyword extraction from titles + PDF metadata keywords
stopwords = {
    "for", "and", "the", "of", "in", "a", "to", "using", "data", "through", "from",
    "on", "with", "by", "an", "public", "transport", "smart", "card", "cards"
}

text_blobs = " ".join(summary_df["paper"].tolist()) + " " + " ".join(pdf_meta_df["keywords_meta"].dropna().astype(str).tolist())
tokens = re.findall(r"[A-Za-z\-]{3,}", text_blobs.lower())
tokens = [t for t in tokens if t not in stopwords]

top_tokens = pd.DataFrame(Counter(tokens).most_common(20), columns=["token", "count"])
top_tokens

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top_tokens["token"], top_tokens["count"], color="#1f77b4")
ax.set_title("Most Frequent Technical Keywords in Paper Set")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
# Build implementation backlog from paper priorities
backlog = (
    summary_df.sort_values(["priority", "year"], ascending=[False, False])
    .assign(
        sprint=np.select(
            [summary_df["priority"] >= 5, summary_df["priority"] == 4],
            ["Sprint 1", "Sprint 2"],
            default="Sprint 3",
        )
    )[["paper", "theme", "pipeline_stage", "priority", "sprint", "project_use"]]
)

backlog

In [ ]:
sprint_map = backlog.groupby(["sprint", "pipeline_stage"], as_index=False).size()
pivot = sprint_map.pivot(index="pipeline_stage", columns="sprint", values="size").fillna(0)

pivot.plot(kind="bar", figsize=(10, 4), colormap="tab20")
plt.title("Research-to-Implementation Roadmap")
plt.ylabel("Number of planned tasks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Practical takeaways for this project

1. Start with **forecast + anomaly baseline** (ASTIR-inspired temporal modeling + event-trigger monitoring).
2. Add **behavioral segmentation** (trip purpose inference, mobility pattern discovery) as second-phase features.
3. Integrate **weather and holiday data** to explain calendar-driven demand variance.
4. Keep an explicit **policy evaluation track** (new line impact, robustness metrics).

This summary notebook can be re-run whenever new papers are added to `docs/papers/`.


---

## Source Notebook: `03_data_science_big_data_sql_foundations.ipynb`

**Section title:** Data Science + Big Data + SQL Foundations (Project Learning Notebook)


# Data Science + Big Data + SQL Foundations (Project Learning Notebook)

Goal: after this notebook, you should be able to read this project confidently, design tables, write SQL queries,
and understand the data-science and big-data workflow behind transport analytics.

## Learning map

1. Data science basics with a transport dataset
2. Big data foundations (why architecture matters)
3. SQL fundamentals (all main actions + operators + symbols)
4. Joins, aggregations, subqueries, CTEs, and window functions
5. PostgreSQL-specific syntax used in this project

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Build a synthetic transport dataset for teaching
dates = pd.date_range("2024-01-01", periods=90, freq="D")

stations = pd.DataFrame(
    {
        "station_id": [101, 102, 201, 202, 301, 302],
        "station_name": [
            "Montparnasse",
            "Bastille",
            "Chatelet",
            "Nation",
            "Times Sq",
            "14 St",
        ],
        "line_code": ["M4", "M5", "M1", "M1", "A", "L"],
        "city": ["Paris", "Paris", "Paris", "Paris", "New York", "New York"],
        "region": ["Ile-de-France", "Ile-de-France", "Ile-de-France", "Ile-de-France", "NYC", "NYC"],
    }
)

ride_rows = []
ride_id = 1
for d in dates:
    dow = d.dayofweek
    weekend_factor = 0.72 if dow >= 5 else 1.0
    for station_id in stations["station_id"]:
        base = 1300 if station_id < 300 else 1800
        seasonal = 1 + 0.15 * np.sin((d.dayofyear / 365) * 2 * np.pi)
        noise = rng.normal(0, 120)
        riders = max(50, int((base * weekend_factor * seasonal) + noise))
        ride_rows.append((ride_id, station_id, d, riders))
        ride_id += 1

rides = pd.DataFrame(ride_rows, columns=["ride_id", "station_id", "ride_date", "riders"])  

weather_rows = []
for d in dates:
    weather_rows.append((d, "Ile-de-France", round(rng.normal(12, 7), 1), round(max(0, rng.gamma(1.7, 1.8)), 1)))
    weather_rows.append((d, "NYC", round(rng.normal(14, 8), 1), round(max(0, rng.gamma(1.9, 2.1)), 1)))
weather = pd.DataFrame(weather_rows, columns=["weather_date", "region", "mean_temp_c", "precip_mm"])

holidays = pd.DataFrame(
    {
        "holiday_date": pd.to_datetime(["2024-01-01", "2024-01-15", "2024-02-19"]),
        "country_code": ["FR", "US", "US"],
        "holiday_name": ["New Year", "MLK Day", "Presidents Day"],
    }
)

rides.head(), stations, weather.head(), holidays

## Part A - Data Science Basics

Data science workflow:

- Understand data shape and meaning
- Clean and validate data
- Explore patterns with statistics and visualization
- Engineer useful features
- Build and evaluate models

In [ ]:
# Basic descriptive analysis
print("rides shape:", rides.shape)
print("stations shape:", stations.shape)
print("weather shape:", weather.shape)

summary = rides["riders"].describe().to_frame("riders_summary")
summary

In [ ]:
# Merge and inspect daily totals
rides_with_station = rides.merge(stations, on="station_id", how="left")
daily_totals = rides_with_station.groupby(["ride_date", "region"], as_index=False)["riders"].sum()

fig, ax = plt.subplots(figsize=(12, 4))
for region, g in daily_totals.groupby("region"):
    ax.plot(g["ride_date"], g["riders"], label=region)
ax.set_title("Daily riders by region")
ax.set_xlabel("Date")
ax.set_ylabel("Riders")
ax.legend()
plt.show()

daily_totals.head()

In [ ]:
# Distribution + spread charts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rides["riders"], bins=25, color="#1f77b4")
axes[0].set_title("Histogram of riders")

box_data = [
    rides_with_station.loc[rides_with_station["region"] == "Ile-de-France", "riders"],
    rides_with_station.loc[rides_with_station["region"] == "NYC", "riders"],
]
axes[1].boxplot(box_data, tick_labels=["Ile-de-France", "NYC"])
axes[1].set_title("Boxplot by region")

merged_weather = daily_totals.merge(weather, left_on=["ride_date", "region"], right_on=["weather_date", "region"], how="left")
axes[2].scatter(merged_weather["precip_mm"], merged_weather["riders"], alpha=0.6)
axes[2].set_title("Rain vs riders")
axes[2].set_xlabel("precip_mm")
axes[2].set_ylabel("riders")

plt.tight_layout()
plt.show()

## Part B - Big Data Basics

Big data is not only "large files". It is usually explained with the **5V** model:

- **Volume**: very large amount of data
- **Velocity**: data arrives quickly (streams)
- **Variety**: many formats (CSV, JSON, logs, text, images)
- **Veracity**: uncertain quality/noise
- **Value**: business impact of extracted insights

In this project:
- MTA hourly data introduces **volume + velocity**
- French + US sources introduce **variety**
- Missing/dirty categories introduce **veracity**
- Forecasting demand and anomalies is the **value**

## Part C - SQL Fundamentals (Actions, Operators, Symbols)

### 1) SQL action families

- **DDL** (schema): `CREATE`, `ALTER`, `DROP`, `TRUNCATE`
- **DML** (data): `SELECT`, `INSERT`, `UPDATE`, `DELETE`, `MERGE`
- **DCL** (permissions): `GRANT`, `REVOKE`
- **TCL** (transactions): `BEGIN`, `COMMIT`, `ROLLBACK`, `SAVEPOINT`

### 2) Core SQL symbols and notations

- `*` all columns
- `;` end statement
- `'text'` string literal
- `"columnName"` quoted identifier
- `%` wildcard (many chars in `LIKE`)
- `_` wildcard (single char in `LIKE`)
- `()` grouping / function arguments
- `--` single-line comment
- `/* ... */` multi-line comment

### 3) Important operators/predicates

- Comparisons: `=`, `<>`, `!=`, `>`, `<`, `>=`, `<=`
- Logical: `AND`, `OR`, `NOT`
- Range/set: `BETWEEN`, `IN`, `NOT IN`
- Pattern: `LIKE`, `ILIKE` (PostgreSQL)
- Null checks: `IS NULL`, `IS NOT NULL`
- Existence: `EXISTS`, `NOT EXISTS`
- PostgreSQL JSON examples: `->`, `->>`, `@>`

In [ ]:
# Create an in-memory SQL database for all runnable SQL examples
conn = sqlite3.connect(":memory:")

stations.to_sql("stations", conn, index=False, if_exists="replace")
rides.to_sql("rides", conn, index=False, if_exists="replace")
weather.to_sql("weather", conn, index=False, if_exists="replace")
holidays.to_sql("holidays", conn, index=False, if_exists="replace")

def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

def exec_sql(query: str) -> None:
    conn.execute(query)
    conn.commit()

run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

In [ ]:
# DDL examples
exec_sql("CREATE TABLE ticket_sales (sale_id INTEGER PRIMARY KEY, station_id INTEGER, amount REAL, sale_date TEXT);")
exec_sql("ALTER TABLE ticket_sales ADD COLUMN payment_method TEXT;")

# Check resulting schema
run_sql("PRAGMA table_info(ticket_sales);")

In [ ]:
# DML examples: INSERT, UPDATE, DELETE, SELECT
exec_sql("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (1, 101, 45.5, '2024-01-01', 'card');")
exec_sql("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (2, 201, 17.0, '2024-01-02', 'cash');")

exec_sql("UPDATE ticket_sales SET amount = amount * 1.1 WHERE payment_method = 'cash';")
exec_sql("DELETE FROM ticket_sales WHERE sale_id = 1;")

run_sql("SELECT * FROM ticket_sales;")

In [ ]:
# WHERE + operators + symbols examples
q = """
SELECT
    ride_id,
    station_id,
    riders
FROM rides
WHERE riders >= 1500
  AND station_id IN (301, 302)
  AND ride_date BETWEEN '2024-01-01' AND '2024-02-15'
ORDER BY riders DESC
LIMIT 10;
"""
run_sql(q)

## Joins (syntax + meaning)

- **INNER JOIN**: only matching rows in both tables
- **LEFT JOIN**: all rows from left table + matches from right table
- **RIGHT JOIN**: all rows from right table + matches from left table (supported in PostgreSQL)
- **FULL OUTER JOIN**: all rows from both tables (supported in PostgreSQL)

Join notation:

```sql
SELECT ...
FROM A
JOIN_TYPE B
  ON A.key = B.key;
```

In [ ]:
# INNER JOIN and LEFT JOIN (runnable)
inner_q = """
SELECT
    r.ride_date,
    s.station_name,
    s.region,
    r.riders
FROM rides r
INNER JOIN stations s
    ON r.station_id = s.station_id
WHERE s.region = 'Ile-de-France'
ORDER BY r.ride_date, s.station_name
LIMIT 10;
"""

left_q = """
SELECT
    d.ride_date,
    d.region,
    d.riders,
    w.mean_temp_c,
    w.precip_mm
FROM (
    SELECT r.ride_date, s.region, SUM(r.riders) AS riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
) d
LEFT JOIN weather w
    ON d.ride_date = w.weather_date
   AND d.region = w.region
ORDER BY d.ride_date, d.region
LIMIT 10;
"""

run_sql(inner_q), run_sql(left_q)

In [ ]:
# RIGHT JOIN / FULL OUTER JOIN in PostgreSQL notation (reference)
postgres_right_join = """
SELECT a.*, b.*
FROM table_a a
RIGHT JOIN table_b b ON a.id = b.id;
"""

postgres_full_join = """
SELECT a.*, b.*
FROM table_a a
FULL OUTER JOIN table_b b ON a.id = b.id;
"""

print("PostgreSQL RIGHT JOIN syntax:", postgres_right_join)
print("PostgreSQL FULL OUTER JOIN syntax:", postgres_full_join)


In [ ]:
# GROUP BY, HAVING, ORDER BY
agg_q = """
SELECT
    s.region,
    s.line_code,
    COUNT(*) AS row_count,
    SUM(r.riders) AS total_riders,
    AVG(r.riders) AS avg_riders
FROM rides r
JOIN stations s ON r.station_id = s.station_id
GROUP BY s.region, s.line_code
HAVING AVG(r.riders) > 1000
ORDER BY total_riders DESC;
"""

agg = run_sql(agg_q)
agg

In [ ]:
# Subquery + CTE + Window function
cte_q = """
WITH daily AS (
    SELECT
        r.ride_date,
        s.region,
        SUM(r.riders) AS total_riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
)
SELECT
    ride_date,
    region,
    total_riders,
    AVG(total_riders) OVER (
        PARTITION BY region
        ORDER BY ride_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7d_avg,
    ROW_NUMBER() OVER (
        PARTITION BY region
        ORDER BY total_riders DESC
    ) AS demand_rank
FROM daily
ORDER BY ride_date, region
LIMIT 20;
"""

cte_df = run_sql(cte_q)
cte_df.head(10)

In [ ]:
# Transaction control (TCL): BEGIN / COMMIT / ROLLBACK
conn.execute("BEGIN")
conn.execute("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (99, 302, 35.0, '2024-01-05', 'card')")
interim = run_sql("SELECT COUNT(*) AS c FROM ticket_sales")

conn.execute("ROLLBACK")
after_rollback = run_sql("SELECT COUNT(*) AS c FROM ticket_sales")

interim, after_rollback

## PostgreSQL-specific syntax you will use in this project

- Auto id: `BIGSERIAL`
- Type cast: `value::numeric`
- Case-insensitive match: `ILIKE '%metro%'`
- Date extraction: `EXTRACT(DOW FROM demand_date)`
- Upsert: `INSERT ... ON CONFLICT (...) DO UPDATE`
- JSONB operators: `->`, `->>`, `@>`
- Copy ingestion: `COPY table FROM 'file.csv' CSV HEADER`

In [ ]:
# Mini project query: demand + weather
mini_q = """
WITH daily AS (
    SELECT
        r.ride_date,
        s.region,
        SUM(r.riders) AS total_riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
)
SELECT
    d.ride_date,
    d.region,
    d.total_riders,
    w.mean_temp_c,
    w.precip_mm,
    CASE WHEN w.precip_mm >= 5 THEN 1 ELSE 0 END AS heavy_rain_flag
FROM daily d
LEFT JOIN weather w
    ON d.ride_date = w.weather_date
   AND d.region = w.region
ORDER BY d.ride_date, d.region;
"""

mini = run_sql(mini_q)
mini.head()

In [ ]:
# Visualization from SQL output
fig, ax = plt.subplots(figsize=(11, 4))
for region, g in mini.groupby("region"):
    ax.plot(pd.to_datetime(g["ride_date"]), g["total_riders"], label=region)
ax.set_title("SQL output: daily riders by region")
ax.set_xlabel("Date")
ax.set_ylabel("Total riders")
ax.legend()
plt.show()

corr = mini[["total_riders", "mean_temp_c", "precip_mm", "heavy_rain_flag"]].corr(numeric_only=True)
corr

## Final checklist

If you can do the following, you are ready for this project:

- Read and clean transport time-series data
- Explain big-data constraints in pipeline design
- Use SQL DDL/DML/TCL confidently
- Write joins, CTEs, aggregations, and window queries
- Understand PostgreSQL syntax needed for production pipeline


---

## Source Notebook: `04_sql_active_learning_practice.ipynb`

**Section title:** SQL Active Learning + Active Recall Practice Notebook


# SQL Active Learning + Active Recall Practice Notebook

Use this notebook repeatedly.

- Active recall cards: retrieve concepts from memory
- Auto-checked SQL tasks: write query, verify result
- Score tracking: monitor your practice progress

In [ ]:
from __future__ import annotations

import sqlite3
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
# Build practice database
dates = pd.date_range("2024-03-01", periods=45, freq="D")

stations = pd.DataFrame(
    {
        "station_id": [1, 2, 3, 4, 5],
        "station_name": ["Alpha", "Bravo", "Charlie", "Delta", "Echo"],
        "line_code": ["L1", "L1", "L2", "L2", "L3"],
        "region": ["FR", "FR", "FR", "US", "US"],
    }
)

rides_rows = []
row_id = 1
for d in dates:
    for sid in stations["station_id"]:
        base = 900 + sid * 90
        weekend = 0.8 if d.dayofweek >= 5 else 1.0
        riders = int(max(30, base * weekend + rng.normal(0, 70)))
        rides_rows.append((row_id, sid, d.strftime("%Y-%m-%d"), riders))
        row_id += 1
rides = pd.DataFrame(rides_rows, columns=["ride_id", "station_id", "ride_date", "riders"])

weather_rows = []
for d in dates:
    weather_rows.append((d.strftime("%Y-%m-%d"), "FR", round(rng.normal(13, 5), 1), round(max(0, rng.gamma(1.6, 1.4)), 1)))
    weather_rows.append((d.strftime("%Y-%m-%d"), "US", round(rng.normal(15, 6), 1), round(max(0, rng.gamma(1.8, 1.7)), 1)))
weather = pd.DataFrame(weather_rows, columns=["weather_date", "region", "temp_c", "rain_mm"])

conn = sqlite3.connect(":memory:")
stations.to_sql("stations", conn, index=False, if_exists="replace")
rides.to_sql("rides", conn, index=False, if_exists="replace")
weather.to_sql("weather", conn, index=False, if_exists="replace")

def run_sql(q: str) -> pd.DataFrame:
    return pd.read_sql_query(q, conn)

run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

## Part 1 - Active Recall Cards

Start with `reveal=False`, answer mentally, then run with `reveal=True`.

In [ ]:
cards = pd.DataFrame(
    [
        ("What does LEFT JOIN return?", "All rows from left table + matching rows from right table."),
        ("Difference between WHERE and HAVING?", "WHERE filters rows before GROUP BY, HAVING filters groups after aggregation."),
        ("What does the % symbol mean in LIKE?", "Wildcard for zero or more characters."),
        ("What is a CTE?", "Common Table Expression: a named temporary result set with WITH."),
        ("What does ROW_NUMBER() do?", "Assigns sequential row numbers inside a partition/order."),
        ("What is NULL in SQL?", "Unknown/missing value; compare with IS NULL, not '='."),
        ("What is COMMIT?", "Makes current transaction changes permanent."),
        ("What is ROLLBACK?", "Reverts current transaction to previous committed state."),
    ],
    columns=["question", "answer"],
)

def draw_cards(n: int = 5, seed: int = 0, reveal: bool = False) -> pd.DataFrame:
    sample = cards.sample(n=min(n, len(cards)), random_state=seed).reset_index(drop=True)
    if reveal:
        return sample
    out = sample.copy()
    out["answer"] = "(hidden)"
    return out

draw_cards(n=5, seed=1, reveal=False)

In [ ]:
# Reveal mode
draw_cards(n=5, seed=1, reveal=True)

## Part 2 - Auto-checked SQL Exercises

Workflow:

1. Read the exercise list
2. Write your SQL in `my_query_X`
3. Run `check_answer(X, my_query_X)`

In [ ]:
exercises = {
    1: {
        "prompt": "Return first 5 rows from rides with columns ride_id, station_id, riders.",
        "solution": "SELECT ride_id, station_id, riders FROM rides ORDER BY ride_id LIMIT 5;",
    },
    2: {
        "prompt": "Count total rows in rides.",
        "solution": "SELECT COUNT(*) AS total_rows FROM rides;",
    },
    3: {
        "prompt": "Average riders per station_id.",
        "solution": "SELECT station_id, AVG(riders) AS avg_riders FROM rides GROUP BY station_id ORDER BY station_id;",
    },
    4: {
        "prompt": "Join rides + stations and return ride_date, station_name, region, riders (first 10 by ride_id).",
        "solution": """
            SELECT r.ride_date, s.station_name, s.region, r.riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            ORDER BY r.ride_id
            LIMIT 10;
        """,
    },
    5: {
        "prompt": "Daily total riders by region.",
        "solution": """
            SELECT r.ride_date, s.region, SUM(r.riders) AS total_riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            GROUP BY r.ride_date, s.region
            ORDER BY r.ride_date, s.region;
        """,
    },
    6: {
        "prompt": "Show only groups where average riders > 1100 (region level).",
        "solution": """
            SELECT s.region, AVG(r.riders) AS avg_riders
            FROM rides r
            JOIN stations s ON r.station_id = s.station_id
            GROUP BY s.region
            HAVING AVG(r.riders) > 1100
            ORDER BY s.region;
        """,
    },
    7: {
        "prompt": "Use a CTE to compute daily total riders, then return top 10 highest-demand days.",
        "solution": """
            WITH daily AS (
                SELECT ride_date, SUM(riders) AS total_riders
                FROM rides
                GROUP BY ride_date
            )
            SELECT *
            FROM daily
            ORDER BY total_riders DESC
            LIMIT 10;
        """,
    },
    8: {
        "prompt": "Use window function to rank days by riders inside each region.",
        "solution": """
            WITH daily AS (
                SELECT r.ride_date, s.region, SUM(r.riders) AS total_riders
                FROM rides r
                JOIN stations s ON r.station_id = s.station_id
                GROUP BY r.ride_date, s.region
            )
            SELECT
                ride_date,
                region,
                total_riders,
                ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_riders DESC) AS demand_rank
            FROM daily
            ORDER BY region, demand_rank;
        """,
    },
}

pd.DataFrame([(k, v["prompt"]) for k, v in exercises.items()], columns=["exercise_id", "prompt"])

In [ ]:
def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out = out.sort_values(list(out.columns)).reset_index(drop=True)
    return out


def check_answer(exercise_id: int, user_sql: str, verbose: bool = True):
    expected = run_sql(exercises[exercise_id]["solution"])
    got = run_sql(user_sql)

    ok = normalize_df(expected).equals(normalize_df(got))
    if verbose:
        print("Exercise", exercise_id)
        print("Status:", "PASS" if ok else "FAIL")
        if not ok:
            print("Expected sample:")
            display(expected.head())
            print("Your sample:")
            display(got.head())
    return ok

In [ ]:
# Example solved check
my_query_1 = "SELECT ride_id, station_id, riders FROM rides ORDER BY ride_id LIMIT 5;"
check_answer(1, my_query_1)

In [ ]:
# TODO: fill your own queries then run check_answer
my_query_2 = "SELECT COUNT(*) AS total_rows FROM rides;"
my_query_3 = "SELECT station_id, AVG(riders) AS avg_riders FROM rides GROUP BY station_id ORDER BY station_id;"
my_query_4 = """
    SELECT r.ride_date, s.station_name, s.region, r.riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    ORDER BY r.ride_id
    LIMIT 10;
"""

quick_results = {
    2: check_answer(2, my_query_2, verbose=False),
    3: check_answer(3, my_query_3, verbose=False),
    4: check_answer(4, my_query_4, verbose=False),
}
quick_results

In [ ]:
# Practice score chart
score_df = pd.DataFrame(
    {
        "exercise_id": list(quick_results.keys()),
        "passed": [int(v) for v in quick_results.values()],
    }
)
score_df["failed"] = 1 - score_df["passed"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(score_df["exercise_id"].astype(str), score_df["passed"], label="pass")
ax.bar(score_df["exercise_id"].astype(str), score_df["failed"], bottom=score_df["passed"], label="fail")
ax.set_title("Current practice score")
ax.set_xlabel("Exercise")
ax.set_ylabel("Result")
ax.legend()
plt.show()

score_df

## Part 3 - Active Learning Loop

Use this loop every day:

1. 5 recall cards (no reveal)
2. 3 SQL exercises (check yourself)
3. Repeat wrong exercises after 2 hours
4. Weekly: run all exercises and record score trend

In [ ]:
# Optional: confidence tracker (self-assessment)
confidence = pd.DataFrame(
    {
        "topic": ["SELECT/WHERE", "JOIN", "GROUP BY/HAVING", "CTE", "WINDOW FUNCTIONS", "TRANSACTIONS"],
        "confidence_1_to_5": [3, 3, 2, 2, 1, 2],
    }
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(confidence["topic"], confidence["confidence_1_to_5"], color="#2ca02c")
ax.set_ylim(0, 5)
ax.set_title("Self-confidence by topic")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

confidence